In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

In [3]:
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .getOrCreate()

In [6]:
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)

In [7]:
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

In [8]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .option("failOnDataLoss", "false") \
    .load()

In [9]:
parsed_stream = kafka_stream \
    .select(
        from_json(
            col("value").cast("string"),
            tx_schema
        ).alias("tx")
    ) \
    .select("tx.*")

In [10]:
fraud_stream = parsed_stream.filter(
    col("amount") > 5000.0
)

In [11]:
enriched_fraud = fraud_stream.join(
    users_df,
    "userId"
)

In [12]:
output_stream = enriched_fraud \
    .withColumn(
        "value",
        to_json(struct("*")).cast("string")
    ) \
    .select("value")

In [13]:
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option("checkpointLocation", "/tmp/fraud_checkpoint") \
    .start()

26/06/20 04:04:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/20 04:04:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/20 04:05:02 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1 : {fraud-notification=UNKNOWN_TOPIC_OR_PARTITION}
                                                                                

In [14]:
query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 